#### 1. Use requests and BeautifulSoup in Python to scrape the latest 10 headlines, authors, publication dates, and summaries from the technology section of a popular news site like NDTV Gadgets or The Verge, and print the extracted data.

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.gadgets360.com/tags/technology/news"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

articles = []

# Find article links
links = soup.select("a[href]")

seen = set()

for link in links:

    title = link.get_text(" ", strip=True)
    article_url = link.get("href")

    if not title or not article_url:
        continue

    if article_url.startswith("/"):
        article_url = "https://www.gadgets360.com" + article_url

    if article_url in seen:
        continue

    try:
        article_response = requests.get(
            article_url,
            headers=headers,
            timeout=10
        )

        article_soup = BeautifulSoup(
            article_response.text,
            "html.parser"
        )

        # Title
        title_tag = article_soup.find("h1")

        if title_tag:
            title = title_tag.get_text(" ", strip=True)

        # Author
        author = "Unknown"

        author_tag = article_soup.select_one(
            '[class*="author"]'
        )

        if author_tag:
            author = author_tag.get_text(" ", strip=True)

        # Publication date
        publication_date = "Unknown"

        date_tag = article_soup.find("time")

        if date_tag:
            publication_date = date_tag.get_text(
                " ",
                strip=True
            )

        # Summary
        summary = ""

        description = article_soup.find(
            "meta",
            attrs={"name": "description"}
        )

        if description:
            summary = description.get("content", "")

        if not summary:
            continue

        articles.append({
            "title": title,
            "author": author,
            "publication_date": publication_date,
            "summary": summary
        })

        seen.add(article_url)

        print("\nTitle:", title)
        print("Author:", author)
        print("Date:", publication_date)
        print("Summary:", summary)
        print("-" * 60)

        if len(articles) == 10:
            break

    except Exception as e:
        continue

#### 2. Store the scraped news data (title, author, publication date, summary) into a pandas DataFrame and save it as a CSV file named tech_news.csv.

In [1]:
import pandas as pd

# Convert scraped articles into DataFrame
df = pd.DataFrame(articles)

# Display DataFrame
print(df)

# Save as CSV
df.to_csv("tech_news.csv", index=False)

print("\nFile saved successfully as tech_news.csv")

NameError: name 'articles' is not defined

#### 3. Load your tech_news.csv file into pandas and perform basic EDA: display the number of articles, count how many articles each author wrote, and show the earliest and latest publication dates.

In [2]:
import pandas as pd

# Load CSV file
df = pd.read_csv("tech_news.csv")

# Number of articles
print("Number of Articles:", len(df))

# Articles written by each author
print("\nArticles by Each Author:")
print(df["author"].value_counts())

# Convert publication date to datetime
df["publication_date"] = pd.to_datetime(
    df["publication_date"],
    errors="coerce"
)

# Earliest publication date
print(
    "\nEarliest Publication Date:",
    df["publication_date"].min()
)

# Latest publication date
print(
    "Latest Publication Date:",
    df["publication_date"].max()
)

FileNotFoundError: [Errno 2] No such file or directory: 'tech_news.csv'

#### 4. Add a feature to your scraper so it skips any article where the summary is missing or empty before saving to CSV.<br><br><em><strong>Hint:</strong> Use an if condition to check for empty summaries during extraction.</em>

In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://www.gadgets360.com/tags/technology/news"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

articles = []

links = soup.select("a[href]")

seen = set()

for link in links:

    title = link.get_text(" ", strip=True)
    article_url = link.get("href")

    if not title or not article_url:
        continue

    if article_url.startswith("/"):
        article_url = "https://www.gadgets360.com" + article_url

    if article_url in seen:
        continue

    try:

        article_response = requests.get(
            article_url,
            headers=headers,
            timeout=10
        )

        article_soup = BeautifulSoup(
            article_response.text,
            "html.parser"
        )

        title_tag = article_soup.find("h1")

        if title_tag:
            title = title_tag.get_text(" ", strip=True)

        author = "Unknown"

        author_tag = article_soup.select_one(
            '[class*="author"]'
        )

        if author_tag:
            author = author_tag.get_text(" ", strip=True)

        date_tag = article_soup.find("time")

        publication_date = "Unknown"

        if date_tag:
            publication_date = date_tag.get_text(
                " ",
                strip=True
            )

        description = article_soup.find(
            "meta",
            attrs={"name": "description"}
        )

        summary = ""

        if description:
            summary = description.get("content", "").strip()

        # Skip article if summary is empty
        if not summary:
            continue

        articles.append({
            "title": title,
            "author": author,
            "publication_date": publication_date,
            "summary": summary
        })

        seen.add(article_url)

        if len(articles) == 10:
            break

    except Exception:
        continue

print("Articles collected:", len(articles))

Articles collected: 0


#### 5. Use ChatGPT to help you write a function that cleans the article summaries by removing extra whitespace and any HTML tags, then apply this function to your DataFrame before saving the CSV.<br><br><em><strong>Hint:</strong> Paste your current summary extraction code into ChatGPT and ask for suggestions on cleaning text data.</em>

In [5]:
import pandas as pd
from bs4 import BeautifulSoup
import re

# Load news CSV
df = pd.read_csv("tech_news.csv")


# Function to clean summaries
def clean_summary(summary):

    if pd.isna(summary):
        return ""

    # Remove HTML tags
    summary = BeautifulSoup(
        str(summary),
        "html.parser"
    ).get_text(" ", strip=True)

    # Remove extra whitespace
    summary = re.sub(
        r"\s+",
        " ",
        summary
    )

    return summary.strip()


# Apply function to summary column
df["summary"] = df["summary"].apply(clean_summary)


# Save cleaned data
df.to_csv(
    "tech_news.csv",
    index=False
)

print("Summaries cleaned successfully.")
print(df[["title", "summary"]])

FileNotFoundError: [Errno 2] No such file or directory: 'tech_news.csv'